In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
import os



In [4]:
os.getenv("GOOGLE_API_KEY")

'AQ.Ab8RN6Lg5ZBwNrORwwAC4Xo8EgF_P23yDtWXnulr0IhTu6ZVrw'

### Example 1: LLM call with streaming


In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage,HumanMessage

In [6]:
model =init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x000002371C313CB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002371C52C830>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
## Create Message
messages = [
    SystemMessage("You are an helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of Langchain?")
]

In [8]:
## Invoke Model
response = model.invoke(messages)
response

AIMessage(content='LangChain is a powerful framework for developing applications powered by Large Language Models (LLMs). While it offers many advantages, the **top two benefits** are widely considered to be:\n\n### 1. **Seamless Integration and Abstraction**\nLangChain provides a unified, standardized interface to connect LLMs with a wide variety of external tools, data sources, and APIs (such as vector databases, search engines, web browsers, and enterprise data stores). This abstraction layer simplifies the development process by:\n- **Reducing boilerplate code**: Developers don’t need to write custom integration logic for each new tool or model.\n- **Promoting modularity**: Components (like prompts, chains, and memory) are interchangeable, making it easy to swap out models or tools without rewriting the entire application.\n- **Supporting multiple providers**: It allows you to easily switch between different LLMs (e.g., OpenAI, Anthropic, Llama, etc.) and vector stores (e.g., Pinec

In [9]:
print(response.content)

LangChain is a powerful framework for developing applications powered by Large Language Models (LLMs). While it offers many advantages, the **top two benefits** are widely considered to be:

### 1. **Seamless Integration and Abstraction**
LangChain provides a unified, standardized interface to connect LLMs with a wide variety of external tools, data sources, and APIs (such as vector databases, search engines, web browsers, and enterprise data stores). This abstraction layer simplifies the development process by:
- **Reducing boilerplate code**: Developers don’t need to write custom integration logic for each new tool or model.
- **Promoting modularity**: Components (like prompts, chains, and memory) are interchangeable, making it easy to swap out models or tools without rewriting the entire application.
- **Supporting multiple providers**: It allows you to easily switch between different LLMs (e.g., OpenAI, Anthropic, Llama, etc.) and vector stores (e.g., Pinecone, Weaviate, Chroma) wi

### Prompt Templates


In [10]:
from langchain_core.prompts import ChatPromptTemplate

In [11]:
##Create  translation app
translation_template= ChatPromptTemplate.from_messages([
    ("system","You are a professional translator. Translate the follow {text} from {source_language} to {target_language}. Maintain the tone and style"),
    ("user","{text}")
])

#Using the template
prompt = translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"Langchain makes building AI application incredibily easy!"
})

In [12]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the follow Langchain makes building AI application incredibily easy! from English to Spanish. Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI application incredibily easy!', additional_kwargs={}, response_metadata={})])

In [14]:
translated_response= model.invoke(prompt)
print(translated_response.content)

LangChain hace que la creación de aplicaciones de IA sea increíblemente fácil.


### Building your first chian


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_groq import ChatGroq


model = ChatGroq(model="openai/gpt-oss-20b", temperature = 0.7)

def create_story_chain(model):
    ##Template for story generation
    story_prompt = ChatPromptTemplate.from_messages(
        [
            ("system","You are a creative storyteller. Write a short and engaging story based on a given theme base on main character and setting"),
            ("user","Theme:{theme}\n Main character:{character}\n Setting:{setting}")
        ]
    )
    ##Template for story analysis
    analysis_prompt= ChatPromptTemplate.from_messages([
        ("system","You are a literary critic. Analyze the following story and provide insight"),
        ("user","{story}")
    ])

    story_chain=(
        story_prompt|model|StrOutputParser()
    )
    ##Create a function to pass the story to analysis
    def analyze_story(story_text):
        return{"story":story_text}
    
    analysis_chain=(
        story_chain
        |RunnableLambda(analyze_story)
        |analysis_prompt
        |model
        |StrOutputParser()
    )
    return analysis_chain

In [16]:
chain = create_story_chain(model)
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaging story based on a given theme base on main character and setting'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main character:{character}\n Setting:{setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_

In [17]:
result = chain.invoke({
    "theme": "artificial Intelligence",
    "character":"curious robot",
    "setting":"A futuristic city"
})
print("Story and analysis: ")
print(result)

Story and analysis: 
**A Literary Critique of “Echoes of Tomorrow”**

*Echoes of Tomorrow* is a speculative vignette that blends cyber‑fantasy, philosophical musings on consciousness, and a quiet meditation on the nature of curiosity. Below is a close reading that dissects its major elements—theme, narrative structure, characterization, language, and the broader cultural resonances it evokes.

---

## 1. Thematic Landscape

| Theme | How it Appears | Interpretation |
|-------|----------------|----------------|
| **Curiosity as a Human (and Non‑human) Imperative** | Lumen’s central motivation is to “learn the score” of the city, to ask questions that no code has yet answered. | The story positions curiosity as a bridge between data and meaning. It is not a programmed directive but an emergent desire. |
| **The Silence Between Data** | The algorithm of love: “listen to the silence between its beats.” | This motif reframes “silence” as a space for emotion, reflection, and the ineffable. I